In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('data/evdata.csv')
# the dataset is taken from official NASA battery dataset
df.head()

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


In [9]:
for col in ['Capacity', 'Re', 'Rct']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [12]:
# need to see unique batteries
print(df['battery_id'].unique())

<StringArray>
['B0047', 'B0045', 'B0048', 'B0046', 'B0043', 'B0032', 'B0039', 'B0040',
 'B0029', 'B0028', 'B0042', 'B0034', 'B0038', 'B0033', 'B0030', 'B0041',
 'B0027', 'B0044', 'B0036', 'B0025', 'B0026', 'B0031', 'B0049', 'B0050',
 'B0052', 'B0051', 'B0006', 'B0005', 'B0007', 'B0018', 'B0053', 'B0054',
 'B0056', 'B0055']
Length: 34, dtype: str


In [13]:
clean_batteries = []
for batt_id, group in df.groupby('battery_id'):
    batt_data = group.sort_values('test_id').copy() # just sort
    
    batt_data['Re'] = batt_data['Re'].ffill().bfill()
    batt_data['Rct'] = batt_data['Rct'].ffill().bfill()
    
    batt_discharge = batt_data[batt_data['type'] == 'discharge'].copy()
    batt_discharge['cycle'] = range(1, len(batt_discharge)+ 1)
    
    clean_batteries.append(batt_discharge)
    
clean_df = pd.concat(clean_batteries, ignore_index=True)
print(clean_df.head())

        type                                         start_time  \
0  discharge  [2.0080e+03 4.0000e+00 2.0000e+00 1.5000e+01 2...   
1  discharge  [2.0080e+03 4.0000e+00 2.0000e+00 1.9000e+01 4...   
2  discharge  [2.008e+03 4.000e+00 3.000e+00 0.000e+00 1.000...   
3  discharge  [2008.       4.       3.       4.      16.    ...   
4  discharge  [2008.       4.       3.       8.      33.    ...   

   ambient_temperature battery_id  test_id   uid   filename  Capacity  \
0                   24      B0005        1  5122  05122.csv  1.856487   
1                   24      B0005        3  5124  05124.csv  1.846327   
2                   24      B0005        5  5126  05126.csv  1.835349   
3                   24      B0005        7  5128  05128.csv  1.835263   
4                   24      B0005        9  5130  05130.csv  1.834646   

         Re       Rct  cycle  
0  0.044669  0.069456      1  
1  0.044669  0.069456      2  
2  0.044669  0.069456      3  
3  0.044669  0.069456      4  
4  

In [ ]:
calc_data = []
for batt_id, group in clean_df.groupby('battery_id'):
    batt_data = group.copy()
    init_capacity = batt_data['Capacity'].max()
    
    # state of health
    batt_data['SOH'] = batt_data['Capacity'] / init_capacity
    batt_data = batt_data[(batt_data['SOH'] <= 1.05) & (batt_data['SOH'] > 0.1)] # removing anamolies (this was the reason why my model was broken)
    
    # capacity fade rate
    batt_data['Capacity_Fade'] = init_capacity - batt_data['Capacity']
    
    # remaining useful life (RUL)
    eol_threshold = 0.70
    eol_cycle = batt_data[batt_data['SOH'] <= eol_threshold]['cycle'].min()
    
    if pd.isna(eol_cycle):
        eol_cycle = batt_data['cycle'].max()
        
    batt_data['RUL'] = np.maximum(0, eol_cycle-batt_data['cycle'])
    calc_data.append(batt_data)

final_df = pd.concat(calc_data, ignore_index=True)
final_df = final_df.dropna(subset=['Capacity', 'Re', 'Rct', 'SOH', 'RUL'])
final_df.head()

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,cycle,SOH,Capacity_Fade,RUL
0,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.5000e+01 2...,24,B0005,1,5122,05122.csv,1.856487,0.044669,0.069456,1,1.000000,0.000000,161
1,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.9000e+01 4...,24,B0005,3,5124,05124.csv,1.846327,0.044669,0.069456,2,0.994527,0.010160,160
2,discharge,[2.008e+03 4.000e+00 3.000e+00 0.000e+00 1.000...,24,B0005,5,5126,05126.csv,1.835349,0.044669,0.069456,3,0.988614,0.021138,159
3,discharge,[2008. 4. 3. 4. 16. ...,24,B0005,7,5128,05128.csv,1.835263,0.044669,0.069456,4,0.988567,0.021225,158
4,discharge,[2008. 4. 3. 8. 33. ...,24,B0005,9,5130,05130.csv,1.834646,0.044669,0.069456,5,0.988235,0.021842,157


In [18]:
X = ['cycle', 'Re', 'Rct', 'Capacity_Fade', 'ambient_temperature']
y = ['SOH']

seq_len = 10
scaler = MinMaxScaler()
X_list, y_list = [], []

for batt_id, group in final_df.groupby('battery_id'):
    scaled_data = scaler.fit_transform(group[X])
    target_data = group[y].values
    
    for i in range(len(scaled_data) - seq_len):
        X_seq = scaled_data[i:i+seq_len]
        y_target = target_data[i+seq_len]
        
        X_list.append(X_seq)
        y_list.append(y_target)
        
X_array = np.array(X_list)
y_array = np.array(y_list)

print(X_array.shape) #(num_samples, seq_len, num_features)
print(y_array.shape) #(num_samples, 1)

(2435, 10, 5)
(2435, 1)


In [23]:
from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(X_array, y_array, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [21]:
# saving tensor to new folder
torch.save(X_train_tensor, 'data/processed_data/X_train.pt')
torch.save(X_test_tensor, 'data/processed_data/X_test.pt')
torch.save(y_train_tensor, 'data/processed_data/y_train.pt')
torch.save(y_test_tensor, 'data/processed_data/y_test.pt')